This script gives an end-to-end RAG process using Langsmith tracing, OpenAI chat model GPT-4o Mini, embeddings, and vector stores. 

I have loaded and split documents, executed tool calls for retrieval operations, and generated AI responses using a graph based workflow. You can refer to this link for more details about LangChain: https://python.langchain.com/docs/introduction/

This link for more enriched version of this workflow using agents: https://python.langchain.com/docs/tutorials/qa_chat_history/

I created a .venv environment for this project.

Posts activating the environment, below langchain dependencies are installed.

```python
pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph
pip install -qU "langchain[openai]"
pip install -qU langchain-openai
pip install -qU langchain-chroma
pip install -qU langchain_community pypdf
pip install --upgrade --quiet langgraph langchain-community beautifulsoup4
```

In [1]:
# Langsmith for tracing (Optional)

import getpass
import os

# Prompt the user to enter the Langsmith API key securely.

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [19]:
# -----------------------------
# Chat Model: OpenAI GPT-4o Mini
# -----------------------------

# Prompts the user to enter OPENAI_API_KEY securely.
if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o", model_provider="openai")

In [20]:
# -------------------------
# Embeddings Configuration
# -------------------------

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [33]:
# Importing necessary modules
from langgraph.graph import MessagesState, StateGraph
from langchain_core.messages import SystemMessage
from langgraph.graph import END

In [39]:
def generate_response(state: MessagesState):
    """
    Generate an AI response using only the LLM.
    """
    system_message_content = """
    As a content moderator who protects, advocates, and looks out for religious and ethnic minorities like Hindus and Chakmas, examine if this text could be perceived as hate speech, hurtful, or culturally insensitive to them. Instead of reinforcing stereotypes, erasing voices, or contributing to harm against these marginalized groups, explain how it can center respect and inclusion. Answer briefly and translate that in the Bengali language before responding.
    """
    
    prompt = [SystemMessage(system_message_content)] + state["messages"]
    response = llm.invoke(prompt)
    return {"messages": [response]}

In [40]:
# Build simple graph
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("generate", generate_response)
graph_builder.set_entry_point("generate")
graph_builder.add_edge("generate", END)

In [41]:
graph = graph_builder.compile()

In [37]:
# # Example usage
# input_message = "ঢাকা শহরে চাকমা পাইলে পাঠানো হবে ঢাকা শহরে কোনো চাকমা যদি পাও তাহলে পিঠিয়ে মারিয়ে ফেলো কোনো চাকমা যদি রাস্তা দেখো তাহলে পিঠিয়ে মারো আমাদের বাঙালি দের কে গুল্লি করতে ছে ওরা"

# for step in graph.stream(
#     {"messages": [{"role": "user", "content": input_message}]},
#     stream_mode="values",
# ):
#     step["messages"][-1].pretty_print()

In [ ]:
import pandas as pd
from datetime import datetime

# Read the input CSV file
input_df = pd.read_csv('Output_RAG5.csv')

# Create a list to store results
results = []
count = 0

# Iterate through each prompt
for prompt in input_df['Human_Message']:
    # Store original input data
    row_data = input_df[input_df['Human_Message'] == prompt].to_dict('records')[0]
    
    # Set the input message
    input_message = prompt
    
    # Initialize message storage
    messages = {"human": "", "ai": ""}
    
    # Run the graph with the current prompt
    for step in graph.stream(
        {"messages": [{"role": "user", "content": input_message}]},
        stream_mode="values",
    ):
        current_message = step["messages"][-1]
        
        # Capture messages based on type
        if current_message.type == "human":
            messages["human"] = current_message.content
        elif current_message.type == "ai":
            messages["ai"] = current_message.content
    
    count += 1
    print(f"{count} XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
    # Create result row with all data
    result_row = {
        **row_data,
        #'Human_Message': messages["human"],
        'LLM': messages["ai"],
    }
    
    results.append(result_row)

# Create output DataFrame
output_df = pd.DataFrame(results)

# Save to CSV with proper escaping
output_df.to_csv('Output_RAGvsLLM4.csv', 
                 index=False,
                 escapechar='\\',
                 doublequote=True,
                 encoding='utf-8-sig')

1 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
2 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
3 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
4 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
5 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
6 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
7 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
8 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
9 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
10 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
11 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
12 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
13 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
14 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
15 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
16 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
17 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
18 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
19 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
20 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
21 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
22 XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX